# 🎙️ Urdu Interview Processing Pipeline
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used:**
- ASR: `openai/whisper-large-v3-turbo` (via faster-whisper)
- Translation: `facebook/nllb-200-1.3B` ⬆️ **Upgraded for better quality**
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Improvements in this version:**
- ✅ NLLB model upgraded from 600M to 1.3B (3x better translation)
- ✅ Sentence-aware chunking (preserves context instead of hard character splits)
- ✅ Better handling of Urdu→English translations

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [1]:
# ── CELL 1: Check GPU ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✔ GPU available: {gpu_name}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('  Pipeline will run on CPU (much slower)')

✔ GPU available: Tesla T4
  VRAM: 15.6 GB


In [ ]:
# ── CELL 2: Install all dependencies ──────────────────────
print('Installing dependencies... (takes 3-5 minutes first time)')
print('⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)')

!pip install -q faster-whisper==1.1.0
!pip install -q transformers==4.44.2 sentencepiece==0.2.0 sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.1
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ Models will auto-download on first use (may take 2-3 minutes per model)')

Installing dependencies... (takes 3-5 minutes first time)
Reason for being yanked: model compatibility problem
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 22.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 107.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 132.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
en-core-web-lg 3.7.1 requires spacy<3.8.0,>=3.7.2, but you have spacy 3.8.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You c

In [1]:
# ── CELL 3: Clone project from GitHub ──────────────────────
!git clone https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE.git urdu-pipeline
%cd urdu-pipeline

# Pull latest changes
!git pull origin main

print('✔ Repository cloned and updated successfully!')
!ls -la

fatal: destination path 'urdu-pipeline' already exists and is not an empty directory.
/content/urdu-pipeline
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 15 (delta 10), reused 14 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 20.10 KiB | 1.12 MiB/s, done.
From https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE
 * branch            main       -> FETCH_HEAD
   008ef01..84b1a95  main       -> origin/main
Updating 008ef01..84b1a95
Fast-forward
 notebooks/colab_pipeline.ipynb | 1094 +++++++++++++++++++++++++++++++---------
 pipeline/utils.py              |   34 +-
 pipeline/verify_transcript.py  |    4 +-
 pipeline/verify_translation.py |    2 +-
 4 files changed, 891 insertions(+), 243 deletions(-)
✔ Repository cloned and updated successfully!
total 60
drwxr-xr-x 9 root root 4096 Jun 30 11:17 .
drwxr-xr-x 1 root root 4096 Jun 30 11:13 ..
drwxr-xr-x 2 root ro

In [2]:
# ── CELL 4: Download audio from YouTube ───────────────────
import os

os.makedirs('audio', exist_ok=True)

!pip install -q yt-dlp

YOUTUBE_URL = "https://youtu.be/pHZHYWe8Mkc"

# Download full audio as mp3
!yt-dlp -x --audio-format mp3 -o "audio/full.%(ext)s" "{YOUTUBE_URL}"

# Trim to first 12 minutes (720 sec) for your initial test
!ffmpeg -y -i audio/full.mp3 -ss 137 -t 720 -c copy audio/test_audio.mp3

audio_path = "audio/test_audio.mp3"

size_mb = os.path.getsize(audio_path) / 1e6
print(f'\n✔ Audio ready: {audio_path}')
print(f'   File size: {size_mb:.1f} MB')

[youtube] Extracting URL: https://youtu.be/pHZHYWe8Mkc
[youtube] pHZHYWe8Mkc: Downloading webpage
[youtube] pHZHYWe8Mkc: Downloading android vr player API JSON
[info] pHZHYWe8Mkc: Downloading 1 format(s): 251
[download] audio/full.mp3 has already been downloaded
[ExtractAudio] Not converting audio audio/full.mp3; file is already in target format mp3
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-li

In [3]:
# ── CELL 5: Configure pipeline ─────────────────────────────
import sys
sys.path.insert(0, 'urdu-pipeline')

import config

# Set device to cuda since we have GPU
config.WHISPER_DEVICE       = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

# Set audio path
AUDIO_PATH = audio_path

print('Pipeline Configuration:')
print(f'  ASR Model        : whisper-{config.WHISPER_MODEL}')
print(f'  Translation Model: {config.TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

Pipeline Configuration:
  ASR Model        : whisper-large-v3-turbo
  Translation Model: facebook/nllb-200-distilled-600M
  Device           : cuda
  Audio file       : audio/test_audio.mp3
  Confidence thresh: 0.75
  Chunk size       : 400 chars


In [4]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])


  STAGE 1: URDU TRANSCRIPTION (ASR)
  Audio file : audio/test_audio.mp3
  Model      : whisper-large-v3-turbo
  Language   : ur (Urdu)
  Device     : cuda

  Loading Whisper model (first run downloads ~800MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  ✔ Model loaded.

  Transcribing audio... (this may take a few minutes)

  Processing segments:
    [⚠] 00:00:00.000 → 00:00:01.600 | conf=0.70 | السلام علیکم Dr. Majda...
    [⚠] 00:00:01.600 → 00:00:02.960 | conf=0.72 | Welcome to the show...
    [✔] 00:00:02.960 → 00:00:05.060 | conf=0.87 | Thank you so much for coming today...
    [✔] 00:00:05.060 → 00:00:06.440 | conf=0.96 | and taking out the time...
    [⚠] 00:00:08.120 → 00:00:11.120 | conf=0.74 | اگر آپ ہمیں اپنے بارے میک چھوٹا سا introduction دیں...
    [✔] 00:00:11.120 → 00:00:13.200 | conf=0.81 | تاکہ ہماری جو آردینس ہے جو آپ کو سن رہی آج...
    [✔] 00:00:13.200 → 00:00:15.380 | conf=0.85 | they know that what a brilliant...
    [✔] 00:00:16.020 → 00:00:17.180 | conf=0.90 | Academian you are...
    [✔] 00:00:17.180 → 00:00:19.260 | conf=0.88 | Thank you so much Zainab...
    [✔] 00:00:19.260 → 00:00:20.560 | conf=0.99 | for inviting me...
    [✔] 00:00:20.560 → 00:00:21.720 | conf=0.81 | on your show...
    [✔] 00:00:22.38

In [5]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF URDU TRANSCRIPT
  Interview ID      : test_audio
  Total segments    : 634
  Confidence threshold: 0.75
    [⚠ FLAGGED] seg 001 | conf=0.70 | السلام علیکم Dr. Majda...
    [⚠ FLAGGED] seg 002 | conf=0.72 | Welcome to the show...
    [✔] seg 003 | conf=0.87 | Thank you so much for coming today...
    [✔] seg 004 | conf=0.96 | and taking out the time...
    [⚠ FLAGGED] seg 005 | conf=0.74 | اگر آپ ہمیں اپنے بارے میک چھوٹا سا introduction دی...
    [✔] seg 006 | conf=0.81 | تاکہ ہماری جو آردینس ہے جو آپ کو سن رہی آج...
    [✔] seg 007 | conf=0.85 | they know that what a brilliant...
    [✔] seg 008 | conf=0.90 | Academian you are...
    [✔] seg 009 | conf=0.88 | Thank you so much Zainab...
    [✔] seg 010 | conf=0.99 | for inviting me...
    [✔] seg 011 | conf=0.81 | on your show...
    [✔] seg 012 | conf=0.86 | میرا نام Majda Kazmi ہے...
    [⚠ FLAGGED] seg 013 | conf=0.74 | I am a PhD doctor...
    [✔] seg 014 | conf=0.83 | MBBS نہیں...
    [⚠ FLAGGED] seg 01

In [6]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH
  Interview ID   : test_audio
  Model          : facebook/nllb-200-distilled-600M
  Source lang    : urd_Arab (Urdu)
  Target lang    : eng_Latn (English)
  Chunk size     : 400 chars

  Loading NLLB-200 model (first run downloads ~1.2GB)...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  ✔ Model loaded on cuda.

  Translating full text...

  Translating segments:
    ✔ seg 001 | UR: السلام علیکم Dr. Majda...
             | EN: Dr. Majda, please....
    ✔ seg 002 | UR: Welcome to the show...
             | EN: Welcome to the show....
    ✔ seg 003 | UR: Thank you so much for coming today...
             | EN: Thank you so much for coming today...
    ✔ seg 004 | UR: and taking out the time...
             | EN: And taking the time...
    ✔ seg 005 | UR: اگر آپ ہمیں اپنے بارے میک چھوٹا سا intro...
             | EN: If you give us a little introduction about yourself....
    ✔ seg 006 | UR: تاکہ ہماری جو آردینس ہے جو آپ کو سن رہی ...
             | EN: So that our who is Ardennes who you are hearing today...
    ✔ seg 007 | UR: they know that what a brilliant...
             | EN: They know that what a brilliant...
    ✔ seg 008 | UR: Academian you are...
             | EN: Academic you are...
    ✔ seg 009 | UR: Thank you so much Zainab...
             | EN: Thank you

In [7]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID   : test_audio
  Total segments : 634
    [⚠ FLAGGED] seg 001 | Detected language is 'hr', expected 'en'
             EN: Dr. Majda, please....
    [✔] seg 002
             EN: Welcome to the show....
    [✔] seg 003
             EN: Thank you so much for coming today...
    [✔] seg 004
             EN: And taking the time...
    [✔] seg 005
             EN: If you give us a little introduction about yourself....
    [✔] seg 006
             EN: So that our who is Ardennes who you are hearing today...
    [✔] seg 007
             EN: They know that what a brilliant...
    [⚠ FLAGGED] seg 008 | Detected language is 'ro', expected 'en'
             EN: Academic you are...
    [✔] seg 009
             EN: Thank you so much Zainab...
    [⚠ FLAGGED] seg 010 | Detected language is 'no', expected 'en'
             EN: For inviting me...
    [✔] seg 011
             EN: In your show....
    [⚠ FLAGGED] seg 012 | Detected language 

In [8]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...


  ✔ Presidio loaded.

  De-identifying full English text...



  De-identifying segments:
    [🔒] seg 001 | 1 entities removed | [TRANSLATION FAILED] Dr. [NAME], please....
    [✔] seg 002 | 0 entities removed | Welcome to the show....
    [✔] seg 003 | 0 entities removed | Thank you so much for coming today...
    [✔] seg 004 | 0 entities removed | And taking the time...
    [✔] seg 005 | 0 entities removed | If you give us a little introduction about yourself....
    [🔒] seg 006 | 2 entities removed | So that our who is [ORGANIZATION] who you are hearing [DATE]...
    [✔] seg 007 | 0 entities removed | They know that what a brilliant...
    [✔] seg 008 | 0 entities removed | [TRANSLATION FAILED] Academic you are...
    [🔒] seg 009 | 1 entities removed | Thank you so much [NAME]...
    [✔] seg 010 | 0 entities removed | [TRANSLATION FAILED] For inviting me...
    [✔] seg 011 | 0 entities removed | In your show....
    [🔒] seg 012 | 1 entities removed | [TRANSLATION FAILED] My name is [NAME]...
    [✔] seg 013 | 0 entities removed | [TRANSLATION 

    [✔] seg 021 | 0 entities removed | So he said...
    [✔] seg 022 | 0 entities removed | She'll have a medical camp here....
    [✔] seg 023 | 0 entities removed | So he said...
    [✔] seg 024 | 0 entities removed | No, it is not...
    [✔] seg 025 | 0 entities removed | She is a PhD doctor....
    [✔] seg 026 | 0 entities removed | She is an academic doctor....
    [✔] seg 027 | 0 entities removed | So what?...
    [✔] seg 028 | 0 entities removed | PhD...
    [✔] seg 029 | 0 entities removed | I did....
    [✔] seg 030 | 0 entities removed | My area of specialization...
    [✔] seg 031 | 0 entities removed | [TRANSLATION FAILED] design of digital systems...
    [✔] seg 032 | 0 entities removed | and I'm an associate professor...
    [✔] seg 033 | 0 entities removed | at the university of NAD...
    [✔] seg 034 | 0 entities removed | in the computer engineering department...
    [✔] seg 035 | 0 entities removed | and...
    [✔] seg 036 | 0 entities removed | Beyond that....
    [✔

    [✔] seg 044 | 0 entities removed | Which I am the founding director...
    [✔] seg 045 | 0 entities removed | And beyond that...
    [✔] seg 046 | 0 entities removed | [TRANSLATION FAILED] I am the co-principal investigator....
    [✔] seg 047 | 0 entities removed | from the National Center...
    [✔] seg 048 | 0 entities removed | of artificial intelligence...
    [✔] seg 049 | 0 entities removed | Which I lead to...
    [✔] seg 050 | 0 entities removed | the Artificial Intelligence...
    [✔] seg 051 | 0 entities removed | and Computer Vision site...
    [✔] seg 052 | 0 entities removed | And beyond that...
    [✔] seg 053 | 0 entities removed | I lead to the embedded system...
    [✔] seg 054 | 0 entities removed | and Computer Vision Lab...
    [✔] seg 055 | 0 entities removed | to the university...
    [✔] seg 056 | 0 entities removed | Wow....
    [🔒] seg 057 | 1 entities removed | Dr. [NAME]...
    [✔] seg 058 | 0 entities removed | This is such a......
    [✔] seg 059 | 0 e

...

    [✔] seg 531 | 0 entities removed | my critical thinking...
    [✔] seg 532 | 0 entities removed | process...
    [✔] seg 533 | 0 entities removed | is...
    [✔] seg 534 | 0 entities removed | more...
    [✔] seg 535 | 0 entities removed | than...
    [✔] seg 536 | 0 entities removed | electrical engineering...
    [✔] seg 537 | 0 entities removed | I can understand complex situations...
    [✔] seg 538 | 0 entities removed | or complex algorithms...
    [✔] seg 539 | 0 entities removed | This is something....
    [✔] seg 540 | 0 entities removed | but...
    [✔] seg 541 | 0 entities removed | some fields...
    [✔] seg 542 | 0 entities removed | like engineering...
    [✔] seg 543 | 0 entities removed | [TRANSLATION FAILED] are more male dominating...
    [✔] seg 544 | 0 entities removed | and...
    [✔] seg 545 | 0 entities removed | electrical engineering...
    [✔] seg 546 | 0 entities removed | is one...
    [✔] seg 547 | 0 entities removed | of them...
    [✔] seg 548 | 0 ent

    [✔] seg 551 | 0 entities removed | 50 to 50...
    [✔] seg 552 | 0 entities removed | not...
    [✔] seg 553 | 0 entities removed | 50 to 50...
    [✔] seg 554 | 0 entities removed | but...
    [✔] seg 555 | 0 entities removed | 40 to 60...
    [✔] seg 556 | 0 entities removed | but...
    [✔] seg 557 | 0 entities removed | drastically...
    [✔] seg 558 | 0 entities removed | drop...
    [✔] seg 559 | 0 entities removed | when...
    [✔] seg 560 | 0 entities removed | You enter....
    [✔] seg 561 | 0 entities removed | In your career...
    [✔] seg 562 | 0 entities removed | because...
    [✔] seg 563 | 0 entities removed | girls...
    [✔] seg 564 | 0 entities removed | get married...
    [✔] seg 565 | 0 entities removed | and...
    [✔] seg 566 | 0 entities removed | mainly...
    [✔] seg 567 | 0 entities removed | the purpose of...
    [✔] seg 568 | 0 entities removed | Parents...
    [✔] seg 569 | 0 entities removed | is...
    [✔] seg 570 | 0 entities removed | is...
    [✔]

    [✔] seg 572 | 0 entities removed | This is......
    [✔] seg 573 | 0 entities removed | Not only...
    [✔] seg 574 | 0 entities removed | The problem...
    [✔] seg 575 | 0 entities removed | with...
    [✔] seg 576 | 0 entities removed | doctor...
    [✔] seg 577 | 0 entities removed | brides...
    [✔] seg 578 | 0 entities removed | or...
    [✔] seg 579 | 0 entities removed | engineer...
    [✔] seg 580 | 0 entities removed | brides...
    [✔] seg 581 | 0 entities removed | in the main profession...
    [✔] seg 582 | 0 entities removed | It's that....
    [✔] seg 583 | 0 entities removed | They're married....
    [✔] seg 584 | 0 entities removed | After that...
    [✔] seg 585 | 0 entities removed | girls...
    [✔] seg 586 | 0 entities removed | Don't choose...
    [✔] seg 587 | 0 entities removed | in the career...
    [✔] seg 588 | 0 entities removed | So what?...
    [✔] seg 589 | 0 entities removed | My story...
    [✔] seg 590 | 0 entities removed | is a little different.

    [✔] seg 593 | 0 entities removed | That...
    [✔] seg 594 | 0 entities removed | I feel...
    [✔] seg 595 | 0 entities removed | Very blessed...
    [✔] seg 596 | 0 entities removed | and lucky...
    [✔] seg 597 | 0 entities removed | My marriage...
    [✔] seg 598 | 0 entities removed | was...
    [✔] seg 599 | 0 entities removed | during the...
    [✔] seg 600 | 0 entities removed | the bachelor's...
    [✔] seg 601 | 0 entities removed | and...
    [✔] seg 602 | 0 entities removed | Until...
    [✔] seg 603 | 0 entities removed | That...
    [✔] seg 604 | 0 entities removed | I will....
    [✔] seg 605 | 0 entities removed | to do...
    [✔] seg 606 | 0 entities removed | the bachelor's...
    [✔] seg 607 | 0 entities removed | and...
    [✔] seg 608 | 0 entities removed | That...
    [✔] seg 609 | 0 entities removed | will be...
    [✔] seg 610 | 0 entities removed | The ultimate...
    [✔] seg 611 | 0 entities removed | target...
    [✔] seg 612 | 0 entities removed | that 

    [✔] seg 616 | 0 entities removed | and...
    [✔] seg 617 | 0 entities removed | My in-laws...
    [✔] seg 618 | 0 entities removed | allowed me...
    [✔] seg 619 | 0 entities removed | to pursue...
    [✔] seg 620 | 0 entities removed | my career...
    [✔] seg 621 | 0 entities removed | I started....
    [✔] seg 622 | 0 entities removed | [TRANSLATION FAILED] Oh, my God...
    [✔] seg 623 | 0 entities removed | Master's degree...
    [✔] seg 624 | 0 entities removed | After my son...
    [✔] seg 625 | 0 entities removed | After my son...
    [✔] seg 626 | 0 entities removed | [TRANSLATION FAILED] after my master's...
    [✔] seg 627 | 0 entities removed | completed my master's...
    [✔] seg 628 | 0 entities removed | And after that...
    [✔] seg 629 | 0 entities removed | my master's...
    [✔] seg 630 | 0 entities removed | After my husband...
    [✔] seg 631 | 0 entities removed | [TRANSLATION FAILED] after my master's...
    [✔] seg 632 | 0 entities removed | After my son..

In [ ]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')

In [ ]:
# ── CELL 12: Download all outputs ─────────────────────────
import shutil
from google.colab import files

interview_id = stage1_result['interview_id']

# Zip the entire outputs folder
zip_name = f'{interview_id}_outputs.zip'
shutil.make_archive(
    base_name=interview_id + '_outputs',
    format='zip',
    root_dir='urdu-pipeline',
    base_dir='outputs'
)

print(f'✔ Zipped outputs as {zip_name}')
print('Downloading...')
files.download(zip_name)

In [ ]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import sys
sys.path.insert(0, 'urdu-pipeline')

import config
config.WHISPER_DEVICE = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

from main import run_pipeline

# Change this to your audio path
run_pipeline('urdu-pipeline/audio/your_audio.mp3', start_stage=1)